# Parallel Coordinates Plots Testing Notebook

This notebook is for manual testing of parallel coordinates plots.
It uses functions from `postprocess_functions.py` and `plot_parcoords_functions.py`.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from ddstartup.postprocessing.postprocess_functions import (
    load_h5_to_dataframe,
    get_input_parameters,
    scale_target,
    apply_filters,
    find_latest_h5_file,
    get_discrete_colorscale
)
from ddstartup.utils.tools import PARAM_UNITS

print("✅ Imports successful")

✅ Imports successful


## Configuration

In [2]:
# Configuration - specify directory or files

# Option 1: Automatic - find latest file in specified directory (DEFAULT)
outputs_dir = Path('../outputs')
files_to_analyze = []

latest_h5_file = find_latest_h5_file(outputs_dir)
if latest_h5_file:
    print(f"📂 Auto-detected folder: {latest_h5_file.parent.name}")
    print(f"📄 Latest file: {latest_h5_file.name}")
    files_to_analyze = [latest_h5_file]

# Option 2: Analyze all files in a specific folder
# outputs_dir = Path('../outputs/20251008_081422_parametric_T_seeded')
# files_to_analyze = sorted(outputs_dir.glob('*.h5'))
# print(f"📂 Using folder: {outputs_dir.name}")
# print(f"📄 Found {len(files_to_analyze)} file(s): {[f.name for f in files_to_analyze]}")

# Option 3: Manually specify exact file(s)
# files_to_analyze = [
#     Path('../outputs/20251008_081422_parametric_T_seeded/parametric_T_seeded.h5'),
#     Path('../outputs/20251007_143015_parametric_lump/parametric_lump.h5'),
# ]
# print(f"📄 Manually specified {len(files_to_analyze)} file(s)")

if files_to_analyze:
    print(f"\n✅ Will analyze {len(files_to_analyze)} file(s)")
else:
    print("⚠️  No files specified")

# Target variables to analyze
target_variables = ['unrealized_gains', 't_startup']

# Optional filters (set to {} for no filtering)
input_filters = {}  # e.g., {'V_plasma': {'min': None, 'max': 150}}
output_filters = {}  # e.g., {'unrealized_gains': {'min': 2e6, 'max': None}}

# Sampling (for performance with large datasets)
max_plot_rows = int(1e6)  # Maximum rows to plot

📂 Auto-detected folder: 20251008_081422_parametric_T_seeded
📄 Latest file: ddstartup_20251008_081422_parametric_T_seeded.h5

✅ Will analyze 1 file(s)


In [3]:
# Loop through all files to analyze
for file_idx, selected_file in enumerate(files_to_analyze, 1):
    print(f"\n{'#'*80}")
    print(f"# FILE {file_idx}/{len(files_to_analyze)}: {selected_file.name}")
    print(f"{'#'*80}\n")
    
    # Load HDF5 data
    print(f"Loading data from {selected_file.name}...")
    df = load_h5_to_dataframe(selected_file)

    print(f"\n📊 Dataset info:")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")
    print(f"\n   Available columns: {list(df.columns)}")


################################################################################
# FILE 1/1: ddstartup_20251008_081422_parametric_T_seeded.h5
################################################################################

Loading data from ddstartup_20251008_081422_parametric_T_seeded.h5...

📊 Dataset info:
   Rows: 256
   Columns: 34

   Available columns: ['E_lost', 'I_target', 'N_ifc', 'N_ofc', 'N_stor', 'P_DDn', 'P_DDp', 'P_DT', 'P_DT_eq', 'P_aux', 'P_aux_DT_eq', 'Q_DD', 'Q_DT_eq', 'TBE', 'TBR_DDn', 'TBR_DT', 'T_i', 'V_plasma', 'capacity_factor', 'cost_of_electricity', 'error', 'eta_th', 'linear_index', 'n_D', 'n_He3', 'n_T', 'n_tot', 'sol_success', 't_startup', 'tau_He3', 'tau_ifc', 'tau_ofc', 'tau_p_T', 'unrealized_gains']


In [4]:
    # Determine file type for plot title
    filename = selected_file.name.lower()
    if "lump" in filename:
        file_type = "lump"
    elif "t_seeded" in filename or "tseeded" in filename:
        file_type = "Tseeded"
    else:
        file_type = "unknown"

    print(f"📝 File type: {file_type}")

📝 File type: Tseeded


In [5]:
    for target in target_variables:
        if target not in df.columns:
            print(f"⚠️  Target '{target}' not found in data. Skipping.")
            continue
        
        print(f"\n{'='*60}")
        print(f"🎯 Target: {target}")
        print(f"{'='*60}")
        
        # Apply filters
        df_filtered = apply_filters(df, input_filters, output_filters, target)
        
        if len(df_filtered) == 0:
            print(f"   ⚠️  No data remaining after filtering. Skipping.")
            continue
        
        # Scale target variable
        df_filtered, target_unit = scale_target(df_filtered, target)
        
        # Get input parameters
        input_parameters = get_input_parameters(df_filtered, target, filename=str(selected_file))
        
        print(f"\n📊 Data after filtering:")
        print(f"   Rows: {len(df_filtered):,}")
        print(f"   Target unit: {target_unit}")
        print(f"   Input parameters: {input_parameters}")
        
        # Sample if too many rows
        if len(df_filtered) > max_plot_rows:
            print(f"   ⚠️  Sampling {max_plot_rows:,} rows for performance")
            df_filtered = df_filtered.sample(n=max_plot_rows, random_state=42)
        
        # Generate parallel coordinates plot (inline)
        print(f"\n🎨 Generating parallel coordinates plot...")
        
        # Color mapping (6 chunks)
        N_COLOR_CHUNKS = 6
        target_values = df_filtered[target]
        quantiles = np.linspace(0, 1, N_COLOR_CHUNKS+1)
        chunk_bounds = target_values.quantile(quantiles).values
        color_indices = np.zeros(len(target_values), dtype=int)
        for i in range(N_COLOR_CHUNKS):
            if i == N_COLOR_CHUNKS-1:
                mask = target_values >= chunk_bounds[i]
            else:
                mask = (target_values >= chunk_bounds[i]) & (target_values < chunk_bounds[i+1])
            color_indices[mask] = i
        
        colorscale = get_discrete_colorscale(N_COLOR_CHUNKS)
        
        # Build dimensions
        dimensions = []
        for param in input_parameters:
            values = df_filtered[param]
            if hasattr(values.iloc[0], "__len__") and not isinstance(values.iloc[0], str):
                continue  # skip vector fields
            
            unique_vals = np.sort(np.unique(values))
            label = f"{param}<br>[{PARAM_UNITS.get(param, '')}]" if param in PARAM_UNITS else param
            
            dim = dict(label=label, values=values, range=[values.min(), values.max()])
            if len(unique_vals) <= 20:
                dim['tickvals'] = unique_vals.tolist()
            dimensions.append(dim)
        
        # Add target dimension
        values = df_filtered[target]
        target_label = f"{target}<br>[{target_unit}]"
        dimensions.append(dict(label=target_label, values=values, range=[values.min(), values.max()]))
        
        # Create figure
        fig = go.Figure(data=go.Parcoords(
            line=dict(
                color=color_indices,
                colorscale=colorscale,
                showscale=True,
                cmin=0,
                cmax=N_COLOR_CHUNKS-1,
                colorbar=dict(
                    title=target_label,
                    thickness=20,
                    len=0.8,
                    tickvals=list(range(N_COLOR_CHUNKS)),
                    ticktext=[f"{chunk_bounds[i]:.2e}–{chunk_bounds[i+1]:.2e}" 
                             for i in range(N_COLOR_CHUNKS)],
                    tickmode='array'
                )
            ),
            dimensions=dimensions
        ))
        
        fig.update_layout(
            title=f"Parallel Coordinates Plot - {file_type} ({target}) - {selected_file.name}",
            font=dict(size=12),
            width=1400,
            height=700,
            margin=dict(l=100, r=120, t=120, b=100),
            paper_bgcolor='white',
            plot_bgcolor='white'
        )
        
        fig.show()
        print(f"✅ Plot displayed")
    
    print(f"\n✅ Completed analysis for {selected_file.name}")


🎯 Target: unrealized_gains
   Filtered: 256 → 224 rows (87.5%)

📊 Data after filtering:
   Rows: 224
   Target unit: M$
   Input parameters: ['V_plasma', 'n_tot', 'T_i', 'tau_p_T', 'P_aux', 'P_aux_DT_eq', 'tau_ifc', 'tau_ofc', 'TBR_DT', 'TBR_DDn', 'eta_th', 'capacity_factor', 'cost_of_electricity']

🎨 Generating parallel coordinates plot...


✅ Plot displayed

🎯 Target: t_startup
   Filtered: 256 → 224 rows (87.5%)

📊 Data after filtering:
   Rows: 224
   Target unit: days
   Input parameters: ['V_plasma', 'n_tot', 'T_i', 'tau_p_T', 'tau_ifc', 'tau_ofc', 'TBR_DT', 'TBR_DDn']

🎨 Generating parallel coordinates plot...


✅ Plot displayed

✅ Completed analysis for ddstartup_20251008_081422_parametric_T_seeded.h5


## Correlation Analysis (Optional)

View correlations between input parameters and target variables.

In [6]:
for target in target_variables:
    if target in df.columns:
        print(f"\n{'='*60}")
        print(f"Correlations with {target}:")
        print(f"{'='*60}")
        
        # Get numeric columns only
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        correlations = df[numeric_cols].corrwith(df[target]).sort_values(ascending=False)
        
        print(correlations.head(10))


Correlations with unrealized_gains:
unrealized_gains       1.000000e+00
E_lost                 9.798307e-01
t_startup              9.120751e-01
tau_ifc                5.071445e-01
TBR_DT                 2.534421e-01
capacity_factor        1.773163e-01
cost_of_electricity    1.069249e-01
eta_th                 8.865813e-02
tau_ofc                1.701793e-02
P_DT_eq               -4.130143e-18
dtype: float64

Correlations with t_startup:
t_startup              1.000000e+00
E_lost                 9.308497e-01
unrealized_gains       9.120751e-01
tau_ifc                5.570852e-01
TBR_DT                 2.513321e-01
tau_ofc                1.051885e-02
P_DT_eq                5.169888e-16
cost_of_electricity    5.167633e-18
capacity_factor        7.239854e-19
eta_th                -7.069816e-18
dtype: float64


/home/alessmor/anaconda3/envs/ddstartupenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning:

invalid value encountered in divide

/home/alessmor/anaconda3/envs/ddstartupenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning:

invalid value encountered in divide

